# Union Export Agent — DGX Spark 一键部署 Notebook

适用：NVIDIA DGX Spark GB10（aarch64, 121GiB 统一内存, CUDA 13.0）

前置：节点可 SSH、有 conda。全部命令来自 2026-09 实测部署路径（12 轮巡检验证）。

## 0. 环境探针（先跑这个）

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!uname -m && python3 -V

## 1. 代码与依赖
pypi.org 受阻时用清华源（实测坑 #1）

In [ ]:
%cd ~
!git clone <repo-url> timo_livekernel 2>/dev/null || echo '已有目录'
%cd timo_livekernel
!pip install -i https://pypi.tuna.tsinghua.edu.cn/simple -r requirements.txt

## 2. occ 环境 + cadquery（STEP 几何）
坑 #2：无 passwordless sudo → 全用户级；坑 #3：conda ToS 先 accept

In [ ]:
!conda create -n occ python=3.11 -y && conda tos accept
!conda run -n occ pip install -i https://pypi.tuna.tsinghua.edu.cn/simple cadquery==2.8.0

## 3. 模型服务（vLLM ×2 实测参数）
Omni-30B 统一端点（LLM/VLM/OCR/ASR 四角色）+ Embed-1B
⚠️ 实测坑 #4：Omni gpu-mem-util 0.55 实占 66GB，与 30B 同驻会 crash-loop（22 次重启的教训）——不要同时驻留 30B 文本档

In [ ]:
!conda run -n nemotron python -m vllm.entrypoints.openai.api_server \
  --model <hf-cache>/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4/files \
  --served-model-name nemotron-omni-30b-a3b --reasoning-parser nemotron_v3 \
  --max-model-len 262144 --gpu-memory-utilization 0.55 \
  --enable-prefix-caching --enable-auto-tool-choice --tool-call-parser qwen3_coder \
  --host 127.0.0.1 --port 8002 &
!sleep 5 && curl -s http://127.0.0.1:8002/v1/models | head -c 200

## 4. 启动主服务 + 自检

In [ ]:
!~/union-deploy/node_services.sh start livekernel 2>/dev/null || \
 conda run -n occ python -m uvicorn services.api_server:app --host 0.0.0.0 --port 8888 &
!sleep 8 && curl -s http://127.0.0.1:8888/health

## 5. 验收（4 项判据 — 全部实测口径）

In [ ]:
import requests, json
B='http://127.0.0.1:8888'
print('health:', requests.get(B+'/health').json()['status'])
r = requests.post(B+'/v1/agent/task', json={'intent':'6061 铝件 阳极氧化 100 件 报价','driver':'console','use_llm':False}, timeout=120).json()
q = (r.get('result') or {}).get('quote') or {}
print('报价:', q.get('unit_price'), q.get('final_price'), '| verify:', (r.get('result') or {}).get('verification_status'))
print('违规:', len(r.get('openshell_violations') or []))
print('RAG:', len(requests.get(B+'/v1/rag/search', params={'q':'6061 阳极氧化','limit':3}, timeout=60).json()['hits']), '命中')

## 6. 回归测试
节点环境注意：26 条失败为环境依赖类（deploy 路径/邮箱状态），核心链路用例全绿

In [ ]:
!conda run -n occ python -m pytest tests/ -q --continue-on-collection-errors 2>&1 | tail -2